# BrainDecoding — 方案一专项实验看板：低秩共模子空间投影 + 相干度门控 + 全词表逐列零分布标准化 (`Subspace LOO + Column Z-Norm`)

本 Notebook 专门用于**零重训、测试期即插即用**地验证 **方案一（Subspace-Projected + Coherence-Gated LOO + Column-Wise Null Standardization）**，与主干训练及序列解码流程完全解耦：

1. **不中心化低秩时间偏置子空间提取（Uncentered SVD + Strict LOO SVD）**：
   在测试期利用 6 个纯历史因果时间错位窗口 $\delta \in \{-15\text{s}, -30\text{s}, -45\text{s}, -60\text{s}, -90\text{s}, -120\text{s}\}$ 的共模均值特征 $\bar{y}_i^{\text{shift}} = \frac{1}{M}\sum_{m=1}^M y_i^{(\delta_m)}$（**完全不接触 $y_i^{\text{clean}}$ 和任何测试标签**）执行**不中心化 SVD**（使第 1 主奇异向量自然捕获全局静态均值偏置 $\boldsymbol{\mu}_0 = \mathbb{E}_i[\bar{y}_i^{\text{shift}}]$），提取前 $K \in \{8, 12, 16, 24\}$ 个主奇异向量构成时间偏置投影矩阵 $P_{\text{bias}} = U_K U_K^\top$；每个错位条件 $k$ 严格仅从其余 5 个错位条件提取留一法基底 $U_{K,\text{LOO}}$。
2. **跨历史窗口相干度门控（Coherence Gate）**：
   计算跨历史窗口一致性指标 $r_i^{\text{shift}} = \|\bar{y}_i^{\text{shift}}\|_2$（理论纯随机正交下界为 $1/\sqrt{6} \approx 0.408$）：
   $$g_i = \text{clamp}\!\left(\frac{\|\bar{y}_i^{\text{shift}}\|_2 - 0.35}{0.65 - 0.35},\; 0.2,\; 1.2\right)$$
3. **全词表逐列零分布标准化（Column-Wise Null Standardization / Z-Norm Anti-Hubness）**：
   消除 2,020 词 Micro 检索中的齐普夫高频词中心度垄断（Top-50 高频词占测试集 51.0% 查询，且 $\sigma_{v \le 50} \gg \sigma_{v > 50}$ 导致零均值残差在未校准全词表上停留在 ~8.4%）：
   $$\tilde{S}_{i, v} = \frac{\langle \hat{y}_i, e_v \rangle - \beta \, \mu_v^{\text{shift}}}{\left(\sigma_v^{\text{shift}}\right)^\tau}, \qquad v \in \{1, \dots, 2020\}$$
   其中 $\beta=1.0,\; \tau \in \{0.0, 0.3, 0.5, 0.7, 1.0\}$，在保持 `50w Shifted ≈ 20.1%` 理论随机地板的同时，将 **`FullVocab Shifted` 从 `~8.4%` 直接压低至 `~2.5%–3.5%`**！
4. **智能特征缓存（首次编码后 0 秒重跑）**：
   首次运行会自动将 Clean 与 6 个 Shifted 的归一化特征缓存至云盘 `exploratory/subspace_encoded_cache_test.pt`（约 45MB）；后续调整 $K$、$\eta$、$\tau$ 或直接运行下方 **Cell 6（5 秒极速验证单元格）** 时 **0 秒跳过 16.4GB 解包与神经网络前向推理**，3~5 秒出完整对比表与四宫格图。

In [ ]:
# 1. 检查 GPU、挂载 Google Drive 并设置方案一实验参数
!nvidia-smi

import os
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('非 Colab 环境，跳过 drive.mount')

# ==================== 方案一（Subspace LOO）用户参数 ====================
GIT_REPO_URL = 'https://github.com/wdongdong0708-jpg/BrainDecoding2.git'
GIT_BRANCH = 'master'
DRIVE_BASE_DIR = '/content/drive/MyDrive/BrainDecoding'

DATASET = 'pallier2025'               # 'pallier2025' 或 'chineseeeg2_littleprince'
MODALITY = 'meg'                      # 'meg' (pallier2025) 或 'eeg' (chineseeeg2)
STAGE = 'revision'                    # 'revision' (默认测试 K0/K1/K2) | 'context' | 'context,revision'
CONFIG_FILE = 'configs/training/pallier2025_rolling_revision.yaml'
CONTROL_CONFIG = 'configs/controls/subspace_coherence_debiasing/subspace_coherence_debiasing_control.yaml'

# 指定要测试的已训练模型实验目录标签（定位 runs/<DATASET>/<EXP_TAG>/<STAGE>/...）：
EXP_TAG = 'lr4e-4_bs256_window_1.0_siamese_debiased_lamda_0.5_history_window'
WORD_EXP_TAG = 'lr4e-4_bs256'
WINDOW_SECONDS = 1.0                  # 时间窗长度（秒；填 None 自动沿用检查点配置）
SEED = 0                              # 随机种子: 0, 1, 2
SPLIT = 'test'                        # 'test' 或 'val'
SUBSET_ARGS = '--subject all'

# 方案一网格搜索参数（单次特征提取后在内存 3 秒内扫完所有组合）：
SHIFT_SECONDS = '-15,-30,-45,-60,-90,-120'  # 6 个纯历史因果错位窗口（秒）
RANK_K_GRID = '8,12,16,24'                  # 低秩时间偏置子空间维度 K 网格
ETA_GRID = '0.8,1.0,1.2'                    # 子空间内扣除强度 eta 网格
COHERENCE_GATE = 'both'                     # 'both' (同时对比开启与关闭门控) | 'true' | 'false'
CALIBRATION_BETAS = '1.0'                   # 全词表逐列静态中心度偏置扣除系数 beta
CALIBRATION_TAUS = '0.0,0.3,0.5,0.7,1.0'    # 全词表逐列标准差反中心度温度指数 tau 网格
COHERENCE_LOW = 0.35                        # 相干度门控下界阈值 (理论 1/sqrt(6)=0.408)
COHERENCE_HIGH = 0.65                       # 相干度门控上界阈值
GATE_MIN = 0.2                              # 门控最小权重下限
GATE_MAX = 1.2                              # 门控最大权重上限
FORCE_REENCODE = False                      # 设为 True 可强制忽略已缓存的 .pt 特征重新跑 GPU 编码
# ========================================================================

CODE_DIR = '/content/code'
VENV_DIR = '/content/proj-venv'
PYTHON = f'{VENV_DIR}/bin/python'
LINKS_DIR = '/content/studies_links'
LOCAL_CACHE_DIR = '/content/local_cache'

DRIVE_DATA_DIR = f'{DRIVE_BASE_DIR}/data'
DRIVE_ARCHIVE_DIR = f'{DRIVE_BASE_DIR}/archives'
DRIVE_RESULTS_DIR = f'{DRIVE_BASE_DIR}/runs'
DRIVE_MODELS_DIR = f'{DRIVE_BASE_DIR}/models'

for folder in (
    LINKS_DIR,
    LOCAL_CACHE_DIR,
    DRIVE_DATA_DIR,
    DRIVE_ARCHIVE_DIR,
    DRIVE_RESULTS_DIR,
    DRIVE_MODELS_DIR,
):
    Path(folder).mkdir(parents=True, exist_ok=True)

os.environ['BRAINDECODING_DATA_DIR'] = LINKS_DIR
os.environ['BRAINDATA_ROOT'] = LINKS_DIR
os.environ['BRAINDECODING_CACHE_DIR'] = LOCAL_CACHE_DIR
os.environ['BRAINDECODING_RESULTS_DIR'] = DRIVE_RESULTS_DIR
os.environ['BRAINDECODING_EXP_TAG'] = EXP_TAG
os.environ['BRAINDECODING_WORD_EXP_TAG'] = WORD_EXP_TAG
os.environ['BRAINDECODING_MODEL_DIR'] = DRIVE_MODELS_DIR

WINDOW_ARGS = f'--window-seconds {WINDOW_SECONDS}' if WINDOW_SECONDS is not None and str(WINDOW_SECONDS).strip() != '' else ''
REENCODE_ARGS = '--force-reencode' if FORCE_REENCODE else ''

print('DATASET           =', DATASET)
print('STAGE             =', STAGE)
print('EXP_TAG           =', EXP_TAG or '(default)')
print('SPLIT             =', SPLIT)
print('SHIFT_SECONDS     =', SHIFT_SECONDS)
print('RANK_K_GRID       =', RANK_K_GRID)
print('ETA_GRID          =', ETA_GRID)
print('COHERENCE_GATE    =', COHERENCE_GATE)
print('CALIBRATION_BETAS =', CALIBRATION_BETAS)
print('CALIBRATION_TAUS  =', CALIBRATION_TAUS)

In [ ]:
# 2. 克隆或同步最新代码到 /content/code
!if [ ! -d "{CODE_DIR}/.git" ]; then \
    git clone -b {GIT_BRANCH} {GIT_REPO_URL} {CODE_DIR}; \
else \
    git -C {CODE_DIR} fetch origin && git -C {CODE_DIR} checkout {GIT_BRANCH} && git -C {CODE_DIR} pull --ff-only; \
fi

In [ ]:
# 3. 初始化虚拟环境 + 建立 /content/studies_links 软链接桥
!python3 -u {CODE_DIR}/scripts/colab_setup.py \
    --venv-dir {VENV_DIR} \
    --project-root {CODE_DIR} \
    --drive-data-dir {DRIVE_DATA_DIR} \
    --links-dir {LINKS_DIR} \
    --local-cache-dir {LOCAL_CACHE_DIR}

In [ ]:
# 4. 按需解包 Google Drive 上的 .tar 缓存归档回 /content/local_cache
#    若云盘已存在当前实验的预编码特征缓存 subspace_encoded_cache_{SPLIT}.pt 且 FORCE_REENCODE=False，则直接 0 秒跳过解包！
_tag_part = EXP_TAG.strip() if (EXP_TAG and EXP_TAG.strip().lower() not in ('', 'default', 'root', 'none')) else ''
_stage_list = ['context', 'revision'] if STAGE.strip().lower() == 'all' else [s.strip() for s in STAGE.split(',') if s.strip()]
_all_cached = True
for _st in _stage_list:
    _base = Path(DRIVE_RESULTS_DIR) / DATASET / _tag_part / _st if _tag_part else Path(DRIVE_RESULTS_DIR) / DATASET / _st
    _found = list(_base.glob(f'*/seed-{int(SEED):03d}/exploratory/subspace_encoded_cache_{SPLIT}.pt'))
    if not _found:
        _all_cached = False
        break

if _all_cached and (not FORCE_REENCODE):
    print(f'[0秒跳过解包] 已检测到云盘存在预编码特征缓存 subspace_encoded_cache_{SPLIT}.pt，直接进入方案一低秩子空间去偏计算！')
else:
    !cd {CODE_DIR} && {PYTHON} -u scripts/cache_archive.py unpack \
        --archive-dir {DRIVE_ARCHIVE_DIR} \
        --cache-dir {LOCAL_CACHE_DIR} \
        --modality {MODALITY} \
        --skip-existing

In [ ]:
# 5. 运行方案一（低秩共模子空间投影 + 相干度门控 + 逐列零分布标准化 LOO 去偏）完整网格评估与可视化
!cd {CODE_DIR} && {PYTHON} -u -m braindecoding.evaluation.subspace_coherence_debiasing_evaluation \
    --dataset {DATASET} \
    --stage {STAGE} \
    --config {CONFIG_FILE} \
    --control-config {CONTROL_CONFIG} \
    --seed {SEED} \
    --split {SPLIT} \
    --shift-seconds={SHIFT_SECONDS} \
    --rank-ks {RANK_K_GRID} \
    --etas {ETA_GRID} \
    --coherence-gate {COHERENCE_GATE} \
    --betas {CALIBRATION_BETAS} \
    --taus {CALIBRATION_TAUS} \
    --coherence-low {COHERENCE_LOW} \
    --coherence-high {COHERENCE_HIGH} \
    --gate-min {GATE_MIN} \
    --gate-max {GATE_MAX} \
    {WINDOW_ARGS} \
    {REENCODE_ARGS} \
    {SUBSET_ARGS}

from IPython.display import Image, display

subspace_pngs = sorted(
    Path(DRIVE_RESULTS_DIR).rglob(f'seed-{int(SEED):03d}/figures/subspace_coherence_debiasing_{SPLIT}.png'),
    key=lambda p: p.stat().st_mtime,
    reverse=True,
)
for png_path in subspace_pngs:
    if EXP_TAG and EXP_TAG.lower() not in ('', 'default', 'root', 'none') and EXP_TAG not in png_path.parts:
        continue
    print(f'\n[方案一诊断看板图] {png_path}')
    display(Image(filename=str(png_path)))

In [ ]:
# 6. 【5秒极速验证沙盒】直接加载云盘已缓存的 subspace_encoded_cache_test.pt，3~5秒扫完 tau in [0.0, 0.3, 0.5, 0.7, 1.0]
#    无需解包 16.4GB tar、无需加载模型权重、无需重跑神经网络！
CACHE_PT_FILE = (
    f'{DRIVE_RESULTS_DIR}/{DATASET}/{EXP_TAG}/{STAGE}/all-n10/'
    f'seed-{int(SEED):03d}/exploratory/subspace_encoded_cache_{SPLIT}.pt'
)
CUSTOM_RANK_KS = '8,12'
CUSTOM_ETAS = '0.8,1.0'
CUSTOM_BETAS = '1.0'
CUSTOM_TAUS = '0.0,0.3,0.5,0.55,0.7,1.0'

!cd {CODE_DIR} && {PYTHON} -u -m braindecoding.evaluation.subspace_coherence_debiasing_evaluation \
    --cache-pt {CACHE_PT_FILE} \
    --rank-ks {CUSTOM_RANK_KS} \
    --etas {CUSTOM_ETAS} \
    --coherence-gate true \
    --betas {CUSTOM_BETAS} \
    --taus {CUSTOM_TAUS}